In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_retries=5,    
)

In [3]:
def chat_node(state: ChatState):
    messages=state['messages']
    response= model.invoke(messages)
    return({'messages': [response]})

In [4]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [8]:
from langchain_core.messages import HumanMessage
initial_state = {
    'messages':[HumanMessage(content="What is Capital of India?")]
}
chatbot.invoke(initial_state)['messages'][-1].content

[{'type': 'text',
  'text': 'The capital of India is **New Delhi**.',
  'extras': {'signature': 'EmAKXgFpFH0TqrTnm33P1YnlTTd570nRD4SmGbT/vv75iBh6ykcM4VvqN5BIKvlRnPKSl3CUS/GjRGig1g/eHAR9rz4XY9GObzG8DaKuK8Ix1MdGH+TZjduaKtGQ6476mdg='}}]

In [9]:
while True:
    user_message = input('Type here: ')

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]})
    print('AI:', response['messages'][-1].content)

AI: [{'type': 'text', 'text': 'The full form of AI is **Artificial Intelligence**.', 'extras': {'signature': 'EmAKXgFpFH0TfsJhjgCX11+QjjV7BeNanzTgQbdnxZJt+cDrbZEfZRlCeGRSdsVzrWAmqhquGiVDwRlD4rN0Hdd4UQNhbadkXufKes6TNFBhWOaPG3J5Tf5W2dM/lSWONNg='}}]
AI: [{'type': 'text', 'text': 'The title **"Godfather of AI"** is most commonly shared by three pioneering computer scientists—**Geoffrey Hinton, Yann LeCun, and Yoshua Bengio**—who won the prestigious **2018 Turing Award** (often considered the "Nobel Prize of Computing") for their foundational breakthroughs in **deep learning**. \n\nHowever, depending on the context, a few other figures are also frequently given this title for different eras of artificial intelligence:\n\n### 1. The Modern Deep Learning Trio (The Turing Award Winners)\n*   **Geoffrey Hinton:** A British-Canadian cognitive psychologist and computer scientist. He is famous for inventing the backpropagation algorithm and Boltzmann machines. *Note: He made headlines in 2023 by r